In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# =============================================================================
# 1. SIMULATION PARAMETERS & TIMELINE (300 Days @ 5-Minute Resolution)
# =============================================================================
np.random.seed(42)  # Ensure reproducibility

days = 300
steps_per_day = 288  # 1440 mins / 5 mins
total_steps = days * steps_per_day
start_date = datetime(2026, 1, 1, 0, 0, 0)
timestamps = [start_date + timedelta(minutes=5 * i) for i in range(total_steps)]

tariff_rate = 209.50       # NGN per kWh (Band A)
ssr_cutoff_watt = 3200.0   # Solid-State Relay Safety Threshold

# =============================================================================
# 2. TEMPORAL & ENVIRONMENTAL FEATURE ENGINEERING
# =============================================================================
df = pd.DataFrame({'timestamp': timestamps})
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['day_of_year'] = df['timestamp'].dt.dayofyear

# Seasonal Weather Profile (Transition from dry hot season to wet season)
day_arr = df['day_of_year'].values
hour_arr = df['hour'].values + df['minute'].values / 60.0

# Ambient Temperature (°C) with diurnal oscillation and seasonal variation
temp_seasonal = 27.0 + 4.0 * np.sin(2 * np.pi * (day_arr - 30) / 365.0)
temp_diurnal = 5.0 * np.sin(2 * np.pi * (hour_arr - 9.0) / 24.0)
df['ambient_temp_c'] = np.round(temp_seasonal + temp_diurnal + np.random.normal(0, 0.7, total_steps), 2)

# Relative Humidity (%) inversely correlated with temperature
humidity_seasonal = 65.0 - 15.0 * np.sin(2 * np.pi * (day_arr - 30) / 365.0)
humidity_diurnal = -10.0 * np.sin(2 * np.pi * (hour_arr - 9.0) / 24.0)
df['relative_humidity_pct'] = np.clip(np.round(humidity_seasonal + humidity_diurnal + np.random.normal(0, 2.0, total_steps), 2), 20.0, 98.0)

# Voltage Profile (Single Phase AC ~220V-240V nominal with grid noise)
df['grid_voltage_v'] = np.round(230.0 + 6.0 * np.sin(2 * np.pi * hour_arr / 24.0) + np.random.normal(0, 2.5, total_steps), 2)

# =============================================================================
# 3. SYNTHETIC APPLIANCE DEMAND & HARDWARE CONTROL ENGINE
# =============================================================================
# Behavioral masks
morning_mask = (hour_arr >= 5.5) & (hour_arr <= 8.0)
evening_mask = (hour_arr >= 18.0) & (hour_arr <= 23.5)
night_mask   = (hour_arr >= 22.0) | (hour_arr <= 5.0)

# 20% Behavioral Drift across 10 months
monthly_drift = np.linspace(0.95, 1.15, total_steps)

# Base continuous refrigeration loads (40% compressor duty cycle)
base_1, base_2, base_3, base_4 = 140.0, 80.0, 60.0, 60.0

# --- Flat 1: Heavy Consumer ---
f1_active = (evening_mask | morning_mask) * 500.0
ac_night = night_mask * (np.random.rand(total_steps) > 0.20) * 1200.0
ac_morning = morning_mask * (np.random.rand(total_steps) > 0.98) * 1200.0  # 2% organic mistake
wh_1 = morning_mask * (np.random.rand(total_steps) > 0.60) * 1500.0
iron_1 = morning_mask * (np.random.rand(total_steps) > 0.80) * 1000.0
micro_1 = evening_mask * (np.random.rand(total_steps) > 0.85) * 1000.0
f1_demanded = (base_1 + f1_active + ac_night + ac_morning + wh_1 + iron_1 + micro_1) * monthly_drift

# --- Flat 2: Normal Consumer 1 ---
f2_active = (evening_mask | morning_mask) * (np.random.rand(total_steps) > 0.10) * 470.0
r_val2 = np.random.rand(total_steps)
iron_2 = morning_mask * ((r_val2 > 0.80) & (r_val2 <= 0.90)) * 1000.0
kettle_2 = morning_mask * (r_val2 > 0.90) * 1200.0
f2_demanded = (base_2 + f2_active + iron_2 + kettle_2) * monthly_drift

# --- Flat 3: Normal Consumer 2 (Shifted Schedule) ---
shifted_morning = (hour_arr >= 7.0) & (hour_arr <= 9.5)
shifted_evening = (hour_arr >= 20.0) | (hour_arr <= 1.0)
f3_active = (shifted_evening | shifted_morning) * (np.random.rand(total_steps) > 0.10) * 470.0
r_val3 = np.random.rand(total_steps)
iron_3 = shifted_morning * ((r_val3 > 0.80) & (r_val3 <= 0.90)) * 1000.0
kettle_3 = shifted_morning * (r_val3 > 0.90) * 1200.0
f3_demanded = (base_3 + f3_active + iron_3 + kettle_3) * monthly_drift

# --- Flat 4: Low Consumer (Minimalist) ---
f4_active = (evening_mask | morning_mask) * (np.random.rand(total_steps) > 0.40) * 350.0
iron_4 = morning_mask * (np.random.rand(total_steps) > 0.95) * 1000.0
f4_demanded = (base_4 + f4_active + iron_4) * monthly_drift

# =============================================================================
# 4. SOLID STATE RELAY (SSR) PROTECTION & TELEMETRY GENERATION
# =============================================================================
demands = [f1_demanded, f2_demanded, f3_demanded, f4_demanded]

for i in range(4):
    flat_num = i + 1
    d_raw = demands[i]
    delivered = np.zeros(total_steps)
    status = np.ones(total_steps, dtype=int)
    cooldown = 0
    
    for t in range(total_steps):
        if cooldown > 0:
            cooldown -= 1
            status[t] = 0
            delivered[t] = 0.0
        elif d_raw[t] > ssr_cutoff_watt:
            status[t] = 0
            delivered[t] = 0.0
            cooldown = 3  # 15-minute cooldown (3 steps)
        else:
            status[t] = 1
            delivered[t] = d_raw[t]
            
    df[f'outlet{flat_num}_demanded_w'] = np.round(d_raw, 2)
    df[f'outlet{flat_num}_delivered_w'] = np.round(delivered, 2)
    df[f'outlet{flat_num}_status'] = status  # 1 = Active, 0 = Tripped/Cutoff
    
    # Electrical Parameter Derivation
    v = df['grid_voltage_v'].values
    pf = np.random.uniform(0.92, 0.98, total_steps)
    current_a = np.where(delivered > 0, delivered / (v * pf), 0.0)
    df[f'outlet{flat_num}_current_a'] = np.round(current_a, 2)
    df[f'outlet{flat_num}_power_factor'] = np.round(pf, 2)
    
    # Cumulative Energy (kWh) and Incremental Step Cost (NGN)
    kwh_step = (delivered * (5.0 / 60.0)) / 1000.0
    df[f'outlet{flat_num}_energy_kwh'] = np.round(np.cumsum(kwh_step), 4)
    df[f'outlet{flat_num}_step_cost_ngn'] = np.round(kwh_step * tariff_rate, 4)

# Aggregate Building Load
df['total_building_power_w'] = (df['outlet1_delivered_w'] + df['outlet2_delivered_w'] + 
                                df['outlet3_delivered_w'] + df['outlet4_delivered_w'])
df['total_building_energy_kwh'] = (df['outlet1_energy_kwh'] + df['outlet2_energy_kwh'] + 
                                   df['outlet3_energy_kwh'] + df['outlet4_energy_kwh'])

# =============================================================================
# 5. EXPORT TO CSV
# =============================================================================
output_filename = 'energenius_energy_dataset.csv'
df.to_csv(output_filename, index=False)
print(f"Dataset successfully exported to '{output_filename}' with shape: {df.shape}")

Dataset successfully exported to 'energenius_energy_dataset.csv' with shape: (86400, 41)


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATASET & AGGREGATE
# =============================================================================
# Load the 5-minute resolution dataset generated previously
df = pd.read_csv('energenius_energy_dataset.csv')

# Aggregate to Daily level for ML Training (to prevent Data Starvation)
daily_df = df.groupby('day_of_year').agg({
    'outlet1_step_cost_ngn': 'sum',
    'ambient_temp_c': 'mean',
    'month': 'first'
}).reset_index()

# Aggregate to Monthly level for Baseline & Final Evaluation
monthly_df = daily_df.groupby('month').agg({
    'outlet1_step_cost_ngn': 'sum'
}).reset_index()

# =============================================================================
# 2. CALCULATE 3-MONTH WMA (BASELINE)
# =============================================================================
W = np.array([0.2, 0.3, 0.5])
wma_predictions = {}
monthly_actuals = monthly_df.set_index('month')['outlet1_step_cost_ngn'].to_dict()

for m in range(4, 11):
    past_3_months = [monthly_actuals[m-3], monthly_actuals[m-2], monthly_actuals[m-1]]
    wma_predictions[m] = np.sum(np.array(past_3_months) * W)

# =============================================================================
# 3. MACHINE LEARNING ENGINE (Train on Daily, Predict Daily, Aggregate Monthly)
# =============================================================================
# Features: [Previous Day Bill, Current Ambient Temp, Day of Month]
daily_df['prev_day_bill'] = daily_df['outlet1_step_cost_ngn'].shift(1).fillna(0)
daily_df['day_of_month'] = daily_df['day_of_year'] % 30

X = daily_df[['prev_day_bill', 'ambient_temp_c', 'day_of_month']].values
Y = daily_df['outlet1_step_cost_ngn'].values

# Define the models
models = {
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    'SVR': SVR(kernel='rbf', C=100, gamma='scale')
}

# We will train on Months 1-3 (Days 1-90), and predict Months 4-10
train_mask = daily_df['month'] <= 3
test_mask = daily_df['month'] >= 4

X_train, Y_train = X[train_mask], Y[train_mask]
X_test, Y_test = X[test_mask], Y[test_mask]
test_months = daily_df['month'][test_mask].values

# Dictionary to store the final monthly aggregated predictions for each model
ml_monthly_predictions = {name: {m: 0 for m in range(4, 11)} for name in models.keys()}

# Train, Predict, and Aggregate
for name, model in models.items():
    # Train
    model.fit(X_train, Y_train)
    # Predict daily
    daily_preds = model.predict(X_test)
    
    # Aggregate daily predictions into monthly totals
    for i, m in enumerate(test_months):
        ml_monthly_predictions[name][m] += daily_preds[i]

# =============================================================================
# 4. EVALUATION METRICS
# =============================================================================
def calculate_metrics(actuals, preds):
    mae = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mape = np.mean(np.abs((actuals - preds) / actuals)) * 100
    r2 = max(0, r2_score(actuals, preds) * 100) # Floor at 0%
    return mae, rmse, mape, r2

target_months = list(range(4, 11))
actual_bills = [monthly_actuals[m] for m in target_months]
wma_bills = [wma_predictions[m] for m in target_months]

# Calculate metrics
results = {}
# WMA Metrics (R2 not applicable for moving averages, assigned N/A)
wma_mae, wma_rmse, wma_mape, _ = calculate_metrics(np.array(actual_bills), np.array(wma_bills))
results['WMA (Baseline)'] = (wma_mae, wma_rmse, wma_mape, "N/A")

# ML Metrics
for name in models.keys():
    preds = [ml_monthly_predictions[name][m] for m in target_months]
    results[name] = calculate_metrics(np.array(actual_bills), np.array(preds))

# =============================================================================
# 5. CONSOLE OUTPUT
# =============================================================================
print("==============================================================================================================")
print("                               TABLE 1: MONTHLY PREDICTIVE BILLING COMPARISON (NGN)                           ")
print("==============================================================================================================")
print(f"{'Month':<8} | {'Actual':<12} | {'WMA':<12} | {'Extra Trees':<12} | {'Random Forest':<13} | {'XGBoost':<12} | {'SVR':<12}")
print("-" * 110)
for m in target_months:
    act = monthly_actuals[m]
    wma = wma_predictions[m]
    etr = ml_monthly_predictions['Extra Trees'][m]
    rf = ml_monthly_predictions['Random Forest'][m]
    xgb = ml_monthly_predictions['XGBoost'][m]
    svr = ml_monthly_predictions['SVR'][m]
    
    print(f"Month {m:<2} | {act:<12.2f} | {wma:<12.2f} | {etr:<12.2f} | {rf:<13.2f} | {xgb:<12.2f} | {svr:<12.2f}")
print("==============================================================================================================\n")

print("=================================================================================")
print("                       TABLE 2: EVALUATION CRITERIA TABLE                        ")
print("=================================================================================")
print(f"{'Algorithm':<15} | {'MAE':<12} | {'RMSE':<12} | {'MAPE (%)':<12} | {'R-Squared (%)':<12}")
print("-" * 81)
for name, metrics in results.items():
    if name == 'WMA (Baseline)':
        print(f"{name:<15} | {metrics[0]:<12.2f} | {metrics[1]:<12.2f} | {metrics[2]:<12.2f} | {metrics[3]:<12}")
    else:
        print(f"{name:<15} | {metrics[0]:<12.2f} | {metrics[1]:<12.2f} | {metrics[2]:<12.2f} | {metrics[3]:<12.2f}")
print("=================================================================================")

                               TABLE 1: MONTHLY PREDICTIVE BILLING COMPARISON (NGN)                           
Month    | Actual       | WMA          | Extra Trees  | Random Forest | XGBoost      | SVR         
--------------------------------------------------------------------------------------------------------------
Month 4  | 105155.00    | 106361.75    | 111739.06    | 110538.38     | 110030.82    | 105528.83   
Month 5  | 108673.16    | 105437.03    | 115692.30    | 114321.33     | 114001.04    | 109030.65   
Month 6  | 105391.70    | 108085.22    | 108579.62    | 108091.34     | 108701.29    | 105516.73   
Month 7  | 110316.09    | 106328.80    | 106222.33    | 106362.34     | 106144.07    | 109123.04   
Month 8  | 113610.50    | 108510.19    | 107945.06    | 107536.53     | 108365.71    | 109248.79   
Month 9  | 109066.02    | 110978.42    | 105773.36    | 105626.61     | 107233.98    | 105705.47   
Month 10 | 96094.44     | 110679.38    | 95330.22     | 95457.06      | 97116.